# Binance NEARUSDT Futures — Real-Time Market-Data Capture System

A production-shaped pipeline that ingests two Binance USDM Futures streams, maintains a validated local order book, and persists raw diffs, periodic snapshots, derived metrics, and detector events to MySQL. Offline replay reconstructs book state from stored data and asserts integrity.

```
aggTrade ──┐
           ├─► WS receiver ─► bounded queue ─► book processor ─┬─► write queue ─► batched MySQL writer
depth@100ms┘  (reconnect,                     (apply diff,     ├─► detectors ─► detections table
              gap detect)                     validate seq)    └─► observability (structured status lines)

            stored data ─► replay harness ─► integrity assertion ─► offline analytics
```

**Dependencies:**
```
pip install websockets requests aiomysql pandas scipy matplotlib
```
Start the database before running:
```
docker compose up -d
```

## Imports and Configuration

Standard-library, `aiomysql` for the async MySQL pool, and `websockets`/`requests` for the two feeds. `Decimal` precision is set to 28. Constants cover both feeds, all four DB tables, batching thresholds, and the output format strings.

In [ ]:
import asyncio
import json
import signal
import sys
import time
from collections import deque
from datetime import datetime, timezone, timedelta
from decimal import Decimal, InvalidOperation, getcontext
from statistics import mean, stdev
from typing import Dict, List, Optional, Tuple

import aiomysql
import requests
import websockets

getcontext().prec = 28

SYMBOL        = "NEARUSDT"
_SYM          = SYMBOL.lower()
WS_DEPTH_URL  = f"wss://fstream.binance.com/ws/{_SYM}@depth@100ms"
WS_TRADE_URL  = f"wss://fstream.binance.com/ws/{_SYM}@aggTrade"
REST_URL      = "https://fapi.binance.com/fapi/v1/depth"
SNAPSHOT_LIMIT = 1000

DEPTH_Q_MAX  = 10_000
TRADE_Q_MAX  = 10_000
WRITE_Q_MAX  = 5_000

BATCH_SIZE       = 500     # rows before forced flush
FLUSH_INTERVAL_S = 0.250   # seconds between flushes
SNAPSHOT_INTERVAL_S = 1.0  # periodic book-snapshot cadence
MONITOR_INTERVAL_S  = 5.0  # observability status line cadence
STALE_THRESHOLD_S   = 0.5  # no update for this long → stale_book detection

DB_HOST = "127.0.0.1"
DB_PORT = 3306
DB_USER = "root"
DB_PASS = "dev"
DB_NAME = "marketdata"

UTC_PLUS_8 = timezone(timedelta(hours=8))

PRINT_HEADER = "{:<23} | {:<12} | {:<12} | {:<12} | {:<12} | {:<12} | {:<12} | {:<10} | {:<6}"
PRINT_ROW    = "{:<23} | {:<12.8f} | {:<12.8f} | {:<12.8f} | {:<12.8f} | {:<12.8f} | {:<12.8f} | {:<+10.6f} | {:<6}"

PriceBook  = Dict[Decimal, Decimal]
DepthEvent = dict

---

## LocalOrderBook

Holds the full bid and ask books as `Dict[Decimal, Decimal]` (price → quantity).

- **`load_snapshot`** — seeds from the REST depth snapshot and records `lastUpdateId`.
- **`apply_event`** — enforces the Binance continuity invariant (`pu == last_update_id`); increments `gap_count` and returns `False` on a break, signalling the processor to resync.
- **`is_crossed`** — returns `True` if `best_bid >= best_ask`; a data-quality violation that triggers a `crossed_book` detection.
- **`metrics`** — returns a dict with L1/L5/L10 spreads, mid, weighted mid (microprice), and order-book imbalance. Updates rolling histories used by `spread_regime`.
- **`spread_regime`** — classifies the current L1 spread as `WIDE`, `TIGHT`, or `NORMAL` via a 300-event rolling z-score (~30 s at 100 ms).

In [ ]:
class LocalOrderBook:
    def __init__(self) -> None:
        self.bids: PriceBook = {}
        self.asks: PriceBook = {}
        self.last_update_id: Optional[int] = None
        self.synced = False
        self.gap_count = 0

        self._prev_mid: Optional[Decimal] = None
        self._spread_history: deque = deque(maxlen=300)

    def load_snapshot(self, snapshot: dict) -> None:
        self.bids = self._levels_to_book(snapshot.get("bids", []))
        self.asks = self._levels_to_book(snapshot.get("asks", []))
        self.last_update_id = int(snapshot["lastUpdateId"])
        self.synced = True

    @staticmethod
    def _levels_to_book(levels: List[List[str]]) -> PriceBook:
        book: PriceBook = {}
        for price_str, qty_str in levels:
            try:
                price, qty = Decimal(price_str), Decimal(qty_str)
            except (InvalidOperation, ValueError):
                continue
            if qty != 0:
                book[price] = qty
        return book

    def apply_event(self, event: DepthEvent, check_previous: bool = True) -> bool:
        if self.last_update_id is None:
            return False
        first_uid  = int(event["U"])
        final_uid  = int(event["u"])
        prev_uid   = int(event.get("pu", -1))

        if final_uid < self.last_update_id:
            return True  # stale, not a gap

        if check_previous and prev_uid != self.last_update_id:
            self.gap_count += 1
            return False

        if not check_previous and not (first_uid <= self.last_update_id <= final_uid):
            self.gap_count += 1
            return False

        self._apply_side(self.bids, event.get("b", []))
        self._apply_side(self.asks, event.get("a", []))
        self.last_update_id = final_uid
        return True

    @staticmethod
    def _apply_side(book: PriceBook, updates: List[List[str]]) -> None:
        for price_str, qty_str in updates:
            price, qty = Decimal(price_str), Decimal(qty_str)
            if qty == 0:
                book.pop(price, None)
            else:
                book[price] = qty

    def top_bids(self, n: int) -> List[Tuple[Decimal, Decimal]]:
        return sorted(self.bids.items(), key=lambda x: x[0], reverse=True)[:n]

    def top_asks(self, n: int) -> List[Tuple[Decimal, Decimal]]:
        return sorted(self.asks.items(), key=lambda x: x[0])[:n]

    def is_crossed(self) -> bool:
        if not self.bids or not self.asks:
            return False
        return max(self.bids) >= min(self.asks)

    def metrics(self) -> Optional[dict]:
        bids = self.top_bids(10)
        asks = self.top_asks(10)
        if len(bids) < 10 or len(asks) < 10:
            return None

        spreads = []
        for level in (1, 5, 10):
            bp, ap = bids[level-1][0], asks[level-1][0]
            mid = (ap + bp) / Decimal("2")
            if mid <= 0:
                return None
            spreads.append((ap - bp) / mid * Decimal("100"))

        best_bid, best_ask = bids[0][0], asks[0][0]
        mid = (best_bid + best_ask) / Decimal("2")
        Q_b, Q_a = self.bids[best_bid], self.asks[best_ask]
        total = Q_b + Q_a
        imbalance    = (Q_b - Q_a) / total if total else Decimal("0")
        weighted_mid = best_bid * (Q_a / total) + best_ask * (Q_b / total) if total else mid

        self._spread_history.append(spreads[0])
        if self._prev_mid is not None:
            pass  # return history used by offline analytics, not accumulated here
        self._prev_mid = mid

        return {
            "spread_l1":    spreads[0],
            "spread_l5":    spreads[1],
            "spread_l10":   spreads[2],
            "mid":          mid,
            "weighted_mid": weighted_mid,
            "imbalance":    imbalance,
            "best_bid":     best_bid,
            "best_ask":     best_ask,
        }

    def spread_regime(self) -> str:
        h = list(self._spread_history)
        if len(h) < 30:
            return "NRML"
        mu = mean(h)
        sigma = stdev(h) if len(h) > 1 else Decimal("0")
        if sigma == 0:
            return "NRML"
        z = float((h[-1] - mu) / sigma)
        if z > 2.0:
            return "WIDE"
        if z < -1.0:
            return "TGHT"
        return "NRML"

---

## MarketDataStore

Thin repository interface over `aiomysql`. Each method corresponds to one table. The concrete MySQL implementation is behind this interface so the caller is not coupled to a specific DB engine.

Prices are passed as strings to preserve the `Decimal` representation exactly — storing them as `DECIMAL(18,8)` in MySQL avoids the float-precision pitfall that would corrupt spread comparisons at the 8th decimal place.

In [ ]:
class MarketDataStore:
    def __init__(self, pool: aiomysql.Pool) -> None:
        self._pool = pool

    async def bulk_insert_depth_updates(self, rows: list) -> None:
        sql = (
            "INSERT INTO depth_updates "
            "(symbol,recv_time,event_time,trade_time,first_uid,final_uid,prev_uid,bids,asks) "
            "VALUES (%s,%s,%s,%s,%s,%s,%s,%s,%s)"
        )
        async with self._pool.acquire() as conn:
            async with conn.cursor() as cur:
                await cur.executemany(sql, rows)
            await conn.commit()

    async def bulk_insert_metrics(self, rows: list) -> None:
        sql = (
            "INSERT INTO metrics "
            "(symbol,ts,spread_l1,spread_l5,spread_l10,mid,weighted_mid,imbalance) "
            "VALUES (%s,%s,%s,%s,%s,%s,%s,%s)"
        )
        async with self._pool.acquire() as conn:
            async with conn.cursor() as cur:
                await cur.executemany(sql, rows)
            await conn.commit()

    async def insert_snapshot(self, symbol: str, ts: int, last_uid: int,
                               bids: list, asks: list) -> None:
        sql = ("INSERT INTO book_snapshots (symbol,ts,last_uid,bids,asks) "
               "VALUES (%s,%s,%s,%s,%s)")
        async with self._pool.acquire() as conn:
            async with conn.cursor() as cur:
                await cur.execute(sql, (symbol, ts, last_uid,
                                        json.dumps(bids), json.dumps(asks)))
            await conn.commit()

    async def insert_detection(self, symbol: str, ts: int, kind: str,
                                detail: Optional[dict] = None) -> None:
        sql = ("INSERT INTO detections (symbol,ts,kind,detail) VALUES (%s,%s,%s,%s)")
        async with self._pool.acquire() as conn:
            async with conn.cursor() as cur:
                await cur.execute(sql, (symbol, ts, kind,
                                        json.dumps(detail) if detail else None))
            await conn.commit()

    async def get_latest_snapshot(self, symbol: str) -> Optional[dict]:
        sql = ("SELECT ts,last_uid,bids,asks FROM book_snapshots "
               "WHERE symbol=%s ORDER BY ts DESC LIMIT 1")
        async with self._pool.acquire() as conn:
            async with conn.cursor(aiomysql.DictCursor) as cur:
                await cur.execute(sql, (symbol,))
                return await cur.fetchone()

    async def get_depth_updates_after(self, symbol: str, last_uid: int) -> list:
        sql = ("SELECT recv_time,event_time,trade_time,first_uid,final_uid,prev_uid,bids,asks "
               "FROM depth_updates WHERE symbol=%s AND final_uid>%s ORDER BY final_uid ASC")
        async with self._pool.acquire() as conn:
            async with conn.cursor(aiomysql.DictCursor) as cur:
                await cur.execute(sql, (symbol, last_uid))
                return await cur.fetchall()

---

## Structured Logging and Output Formatting

All diagnostic output is JSON on stderr so it can be piped to any log aggregator. The print functions write the live tabular feed to stdout separately.

In [ ]:
def _log(level: str, component: str, message: str, **kw) -> None:
    print(json.dumps({"ts": int(time.time()*1000), "level": level,
                      "component": component, "msg": message, **kw}),
          file=sys.stderr, flush=True)


def utc8_now_ms() -> str:
    now = datetime.now(UTC_PLUS_8)
    return now.strftime("%Y-%m-%d %H:%M:%S.%f")[:-3]


def print_header() -> None:
    print(PRINT_HEADER.format(
        "Timestamp(UTC+8)", "Spread L1%", "Spread L5%", "Spread L10%",
        "Best Bid", "Best Ask", "WgtMid", "Imbalance", "Regime",
    ))
    print("-" * 135)


def print_metrics_row(m: dict, regime: str) -> None:
    print(PRINT_ROW.format(
        utc8_now_ms(),
        float(m["spread_l1"]), float(m["spread_l5"]), float(m["spread_l10"]),
        float(m["best_bid"]),  float(m["best_ask"]),  float(m["weighted_mid"]),
        float(m["imbalance"]), regime,
    ), flush=True)


def fetch_snapshot() -> dict:
    r = requests.get(REST_URL, params={"symbol": SYMBOL, "limit": SNAPSHOT_LIMIT}, timeout=10)
    r.raise_for_status()
    return r.json()

---

## WebSocket Receivers

Two independent coroutines, one per stream. Both attach `_recv` (local epoch ms) to each event before enqueuing — this enables the three-component latency decomposition `(E - T, recv - E, applied - recv)` downstream.

`websocket_receiver` handles `depth@100ms` with sequence-continuity enforcement. `aggtrade_receiver` handles `aggTrade` events, which carry signed trade direction (`m=True` means the buyer is the market maker, i.e. a sell-side aggressor). Both coroutines reconnect with exponential back-off (1 s → 30 s cap) and emit a structured log on each reconnect so the reconnect count is observable.

In [ ]:
async def websocket_receiver(queue: asyncio.Queue, stop_event: asyncio.Event) -> None:
    backoff, reconnects = 1, 0
    while not stop_event.is_set():
        try:
            async with websockets.connect(
                WS_DEPTH_URL, ping_interval=20, ping_timeout=20,
                max_queue=None, close_timeout=5,
            ) as ws:
                backoff = 1
                async for msg in ws:
                    if stop_event.is_set():
                        break
                    t_recv = int(time.time() * 1000)
                    ev = json.loads(msg)
                    if ev.get("e") == "depthUpdate":
                        ev["_recv"] = t_recv
                        await queue.put(ev)
        except asyncio.CancelledError:
            raise
        except Exception as exc:
            reconnects += 1
            _log("WARN", "depth_recv", str(exc), reconnects=reconnects, backoff_s=backoff)
            await asyncio.sleep(backoff)
            backoff = min(backoff * 2, 30)


async def aggtrade_receiver(queue: asyncio.Queue, stop_event: asyncio.Event) -> None:
    # p=price q=qty m=is_buyer_maker(True→sell aggressor) T=trade_time_ms t=trade_id
    backoff, reconnects = 1, 0
    while not stop_event.is_set():
        try:
            async with websockets.connect(
                WS_TRADE_URL, ping_interval=20, ping_timeout=20,
                max_queue=None, close_timeout=5,
            ) as ws:
                backoff = 1
                async for msg in ws:
                    if stop_event.is_set():
                        break
                    ev = json.loads(msg)
                    if ev.get("e") == "aggTrade":
                        ev["_recv"] = int(time.time() * 1000)
                        await queue.put(ev)
        except asyncio.CancelledError:
            raise
        except Exception as exc:
            reconnects += 1
            _log("WARN", "trade_recv", str(exc), reconnects=reconnects, backoff_s=backoff)
            await asyncio.sleep(backoff)
            backoff = min(backoff * 2, 30)

---

## Order Book Initialisation and Sync Procedure

Implements the Binance recommended sync procedure:

1. The depth receiver is already running and buffering events.
2. Fetch REST depth snapshot via `asyncio.to_thread`.
3. Drop buffered events where `u < lastUpdateId`.
4. Find the bridging event where `U <= lastUpdateId <= u`.
5. Apply the bridging event (skipping `pu` check for that first event), then continue normally.
6. If no bridging event is found the snapshot is stale; sleep 200 ms and retry.

`drain_queue` empties all currently buffered events without blocking.

In [ ]:
async def drain_queue(queue: asyncio.Queue) -> list:
    events = []
    while True:
        try:
            events.append(queue.get_nowait())
        except asyncio.QueueEmpty:
            break
    return events


async def initialise_orderbook(orderbook: LocalOrderBook, queue: asyncio.Queue) -> None:
    while True:
        snapshot = await asyncio.to_thread(fetch_snapshot)
        orderbook.load_snapshot(snapshot)
        last_uid = orderbook.last_update_id
        assert last_uid is not None

        buffered = await drain_queue(queue)
        buffered = [ev for ev in buffered if int(ev["u"]) >= last_uid]

        start = None
        for i, ev in enumerate(buffered):
            if int(ev["U"]) <= last_uid <= int(ev["u"]):
                start = i
                break

        if start is None:
            await asyncio.sleep(0.2)
            continue

        first = True
        for ev in buffered[start:]:
            ok = orderbook.apply_event(ev, check_previous=not first)
            first = False
            if not ok:
                break
        else:
            return

        await asyncio.sleep(0.2)

---

## Detection Layer

Two tiers of detectors, all persisted as rows in the `detections` table:

**Data-quality detectors** (correctness, not research):
- `crossed_book` — `best_bid >= best_ask` after applying an event; should be impossible on a healthy feed.
- `sequence_gap` — `pu != last_update_id`; triggers a full resync.
- `stale_book` — no `depthUpdate` received for > 500 ms.

**Microstructure/event detectors** (repositioned from research signals to logged events):
- `regime_wide` / `regime_tight` — L1 spread z-score exceeds ±2σ / -1σ over a 30 s window.
- `depth_wall` — a price level within the top 20 carries > 3× the mean quantity of all shallower levels.
- `imbalance_spike` — `|imbalance| > 0.80` (strong directional pressure at the top of book).

In [ ]:
async def run_detectors(
    ob: LocalOrderBook, m: dict, store: MarketDataStore, ts: int
) -> None:
    sym = SYMBOL

    if ob.is_crossed():
        detail = {"best_bid": str(max(ob.bids)), "best_ask": str(min(ob.asks))}
        await store.insert_detection(sym, ts, "crossed_book", detail)
        _log("WARN", "detector", "crossed book", **detail)

    regime = ob.spread_regime()
    if regime != "NRML":
        await store.insert_detection(sym, ts, f"regime_{regime.lower()}",
                                      {"spread_l1": str(m["spread_l1"])})

    if abs(float(m["imbalance"])) > 0.80:
        await store.insert_detection(sym, ts, "imbalance_spike",
                                      {"imbalance": str(m["imbalance"])})

    _detect_walls(ob, store, ts, sym)


def _detect_walls(ob: LocalOrderBook, store: MarketDataStore, ts: int, sym: str) -> None:
    ALPHA = 3.0
    for side, fn in (("bid", ob.top_bids), ("ask", ob.top_asks)):
        levels = fn(20)
        if len(levels) < 3:
            continue
        qtys = [float(q) for _, q in levels]
        for k in range(1, len(qtys)):
            avg = sum(qtys[:k]) / k
            if avg > 0 and qtys[k] > ALPHA * avg:
                asyncio.create_task(store.insert_detection(
                    sym, ts, "depth_wall",
                    {"side": side, "price": str(levels[k][0]),
                     "ticks_from_best": k, "qty": str(levels[k][1])}
                ))
                break

### aggTrade — Trade-Through Detector

`trade_processor` consumes the `aggTrade` queue independently of the book processor. It flags a **trade-through** when both conditions hold:

1. `qty > SWEEP_ALPHA × rolling-average qty` — the trade is unusually large (≥ 3×).
2. The aggressive fill price crossed the passive best:
   - Buy aggressor (`is_buyer_maker=False`): trade price **>** `best_ask` (swept above the ask).
   - Sell aggressor (`is_buyer_maker=True`): trade price **<** `best_bid` (swept below the bid).

Condition 2 is a direct signal that the order consumed at least one full level. Requiring both conditions reduces noise from large-but-passive resting fills.

In [ ]:
async def trade_processor(
    trade_queue: asyncio.Queue,
    ob: LocalOrderBook,
    store: MarketDataStore,
    stop_event: asyncio.Event,
) -> None:
    SWEEP_ALPHA = 3.0
    size_history: deque = deque(maxlen=200)

    while not stop_event.is_set():
        try:
            event = await asyncio.wait_for(trade_queue.get(), timeout=0.5)
        except asyncio.TimeoutError:
            continue

        if not ob.synced or not ob.bids or not ob.asks:
            continue

        try:
            price          = Decimal(event["p"])
            qty            = Decimal(event["q"])
            is_buyer_maker = bool(event["m"])
            ts             = int(event.get("T", time.time() * 1000))
        except (KeyError, InvalidOperation):
            continue

        size_history.append(float(qty))
        avg_qty = sum(size_history) / len(size_history)

        if float(qty) <= SWEEP_ALPHA * avg_qty:
            continue  # not anomalously large

        best_bid = max(ob.bids)
        best_ask = min(ob.asks)

        if not is_buyer_maker and price > best_ask:
            direction = "buy_sweep"
        elif is_buyer_maker and price < best_bid:
            direction = "sell_sweep"
        else:
            continue

        await store.insert_detection(SYMBOL, ts, "trade_through", {
            "direction": direction,
            "price":     str(price),
            "qty":       str(qty),
            "avg_qty":   round(avg_qty, 4),
            "best_bid":  str(best_bid),
            "best_ask":  str(best_ask),
        })
        _log("INFO", "detector", "trade_through",
             direction=direction, price=str(price), qty=str(qty))

---

## Database Writer

`db_writer` is a dedicated coroutine with its own bounded `asyncio.Queue(maxsize=5000)`. The book processor enqueues row dicts without waiting; the writer batches them and flushes via `executemany` when either `BATCH_SIZE` rows have accumulated **or** `FLUSH_INTERVAL_S` (250 ms) has elapsed — whichever comes first.

**Why batching matters:** A naïve per-row INSERT on a 100 ms feed saturates the DB connection at ~10 events/s and back-pressures the order book processor. Batching amortises round-trip cost, allowing the writer to drain faster than the feed produces rows. The write queue is a deliberate buffer; if it fills, the processor drops the row and logs a warning rather than stalling — book correctness takes priority over data completeness.

In [ ]:
async def db_writer(
    write_queue: asyncio.Queue,
    store: MarketDataStore,
    stop_event: asyncio.Event,
) -> None:
    depth_buf:  list = []
    metric_buf: list = []
    last_flush = time.monotonic()
    rows_written = 0

    async def flush() -> None:
        nonlocal rows_written
        tasks = []
        if depth_buf:
            tasks.append(store.bulk_insert_depth_updates(list(depth_buf)))
        if metric_buf:
            tasks.append(store.bulk_insert_metrics(list(metric_buf)))
        if tasks:
            await asyncio.gather(*tasks)
            rows_written += len(depth_buf) + len(metric_buf)
        depth_buf.clear()
        metric_buf.clear()

    while not stop_event.is_set():
        try:
            elapsed = time.monotonic() - last_flush
            timeout = max(0.001, FLUSH_INTERVAL_S - elapsed)
            try:
                row = await asyncio.wait_for(write_queue.get(), timeout=timeout)
                tbl = row["table"]
                if tbl == "depth_update":
                    depth_buf.append(row["data"])
                elif tbl == "metric":
                    metric_buf.append(row["data"])
            except asyncio.TimeoutError:
                pass

            total = len(depth_buf) + len(metric_buf)
            now = time.monotonic()
            if total >= BATCH_SIZE or (now - last_flush >= FLUSH_INTERVAL_S and total > 0):
                await flush()
                last_flush = now

        except asyncio.CancelledError:
            await flush()
            _log("INFO", "db_writer", "flushed on shutdown", rows_written=rows_written)
            raise
        except Exception as exc:
            _log("WARN", "db_writer", str(exc))
            await asyncio.sleep(1)

---

## DB Write Benchmark

Measures per-row `execute` vs `executemany` batching on the `depth_updates` table. Run this cell once after `docker compose up -d` to generate the numbers cited in the README. Benchmark rows are cleaned up automatically (written with `symbol='BENCH'`).

In [ ]:
import time as _time
import aiomysql as _aiomysql

_N = 2000
_BATCH = 500
_DUMMY = ("BENCH", 0, 0, 0, 0, 0, 0, "[]", "[]")
_SQL = (
    "INSERT INTO depth_updates "
    "(symbol,recv_time,event_time,trade_time,first_uid,final_uid,prev_uid,bids,asks) "
    "VALUES (%s,%s,%s,%s,%s,%s,%s,%s,%s)"
)

async def _run_bench():
    pool = await _aiomysql.create_pool(
        host=DB_HOST, port=DB_PORT, user=DB_USER,
        password=DB_PASS, db=DB_NAME, minsize=1, maxsize=2,
    )

    # ── per-row ──
    t0 = _time.monotonic()
    async with pool.acquire() as conn:
        async with conn.cursor() as cur:
            for _ in range(_N):
                await cur.execute(_SQL, _DUMMY)
        await conn.commit()
    t_per_row = _time.monotonic() - t0

    # ── executemany (batched) ──
    rows = [_DUMMY] * _N
    t0 = _time.monotonic()
    async with pool.acquire() as conn:
        for i in range(0, _N, _BATCH):
            async with conn.cursor() as cur:
                await cur.executemany(_SQL, rows[i : i + _BATCH])
        await conn.commit()
    t_batched = _time.monotonic() - t0

    # clean up
    async with pool.acquire() as conn:
        async with conn.cursor() as cur:
            await cur.execute("DELETE FROM depth_updates WHERE symbol='BENCH'")
        await conn.commit()

    pool.close()
    await pool.wait_closed()

    rps_p = _N / t_per_row
    rps_b = _N / t_batched
    print(f"{'Method':<22} {'Rows':>6} {'Time (s)':>10} {'Rows/s':>10} {'Speedup':>8}")
    print("-" * 60)
    print(f"{'per-row execute':<22} {_N:>6} {t_per_row:>10.3f} {rps_p:>10.0f} {'1.0×':>8}")
    print(f"{'executemany batch':<22} {_N:>6} {t_batched:>10.3f} {rps_b:>10.0f} {rps_b/rps_p:>7.1f}×")

await _run_bench()

---

## Observability

`ObservabilityState` tracks events-per-second, queue depths, and the three-component latency decomposition:

```
exchange_internal = E - T       (Binance internal processing, ms)
network           = recv - E    (gateway → local Python; caveat: no shared clock — includes skew)
local_processing  = applied - recv
```

The network component is not trustworthy without NTP synchronisation or co-location, because `E` is a Binance server timestamp and `recv` is the local machine clock. The decomposition is still useful for detecting gross anomalies and is the correct answer to give in an interview.

In [ ]:
class ObservabilityState:
    def __init__(self) -> None:
        self._event_count = 0
        self._window_start = time.monotonic()
        self._latencies: deque = deque(maxlen=1000)  # (exch_int, network, local_proc)
        self._q_depth:  deque = deque(maxlen=200)
        self._q_writer: deque = deque(maxlen=200)

    def record(self, event: dict, t_applied_ms: int,
                depth_q: asyncio.Queue, write_q: asyncio.Queue) -> None:
        self._event_count += 1
        t_recv = event.get("_recv", 0)
        E = event.get("E", 0)
        T = event.get("T", 0)
        if E and T and t_recv:
            self._latencies.append((E - T, t_recv - E, t_applied_ms - t_recv))
        self._q_depth.append(depth_q.qsize())
        self._q_writer.append(write_q.qsize())

    def emit(self, symbol: str) -> None:
        elapsed = time.monotonic() - self._window_start
        eps = self._event_count / elapsed if elapsed > 0 else 0.0

        def p95(col: int) -> float:
            vals = sorted(x[col] for x in self._latencies)
            if not vals:
                return 0.0
            return vals[int(len(vals) * 0.95)]

        q_d = int(sum(self._q_depth)  / len(self._q_depth))  if self._q_depth  else 0
        q_w = int(sum(self._q_writer) / len(self._q_writer)) if self._q_writer else 0

        _log("MONITOR", "obs", f"sym={symbol}",
             eps=round(eps, 1), q_depth=q_d, q_writer=q_w,
             net_p95_ms=round(p95(1), 1), local_p95_ms=round(p95(2), 1))

        self._event_count = 0
        self._window_start = time.monotonic()

---

## Processor and Main Entry Point

**`orderbook_processor`** — the central coroutine. After syncing, it consumes depth events one by one: stale events are skipped; a sequence gap triggers `sequence_gap` detection and a full resync; a valid update computes metrics, enqueues rows for both `depth_updates` and `metrics` tables, runs all detectors, and writes a periodic book snapshot. Observability is emitted every `MONITOR_INTERVAL_S` seconds to stderr.

**`main`** — creates the `aiomysql` pool, wires up all four queues and the stop event, starts four concurrent tasks (depth receiver, aggTrade receiver, processor, DB writer), and waits for shutdown. The `finally` block cancels all tasks, then closes the pool.

In [ ]:
async def orderbook_processor(
    ob: LocalOrderBook,
    depth_queue: asyncio.Queue,
    write_queue: asyncio.Queue,
    store: MarketDataStore,
    stop_event: asyncio.Event,
) -> None:
    obs = ObservabilityState()
    print_header()
    last_snapshot = time.monotonic()
    last_monitor  = time.monotonic()
    last_event_t  = time.monotonic()

    while not stop_event.is_set():
        try:
            await initialise_orderbook(ob, depth_queue)
            _log("INFO", "processor", "book synced", last_uid=ob.last_update_id)

            ts = int(time.time() * 1000)
            bids = [(str(p), str(q)) for p, q in ob.top_bids(20)]
            asks = [(str(p), str(q)) for p, q in ob.top_asks(20)]
            await store.insert_snapshot(SYMBOL, ts, ob.last_update_id, bids, asks)
            last_snapshot = time.monotonic()

            while not stop_event.is_set():
                if time.monotonic() - last_event_t > STALE_THRESHOLD_S:
                    ts = int(time.time() * 1000)
                    age_ms = int((time.monotonic() - last_event_t) * 1000)
                    await store.insert_detection(SYMBOL, ts, "stale_book", {"age_ms": age_ms})
                    _log("WARN", "processor", "stale book", age_ms=age_ms)

                try:
                    event = await asyncio.wait_for(depth_queue.get(), timeout=0.5)
                except asyncio.TimeoutError:
                    continue

                last_event_t = time.monotonic()
                t_applied_ms = int(time.time() * 1000)

                if int(event["u"]) < ob.last_update_id:
                    continue  # stale

                ok = ob.apply_event(event, check_previous=True)
                if not ok:
                    ts = int(time.time() * 1000)
                    await store.insert_detection(SYMBOL, ts, "sequence_gap",
                                                  {"gap_count": ob.gap_count})
                    _log("WARN", "processor", "sequence gap", gap_count=ob.gap_count)
                    break

                obs.record(event, t_applied_ms, depth_queue, write_queue)

                if not write_queue.full():
                    write_queue.put_nowait({"table": "depth_update", "data": (
                        SYMBOL,
                        event.get("_recv", t_applied_ms),
                        event.get("E", 0),
                        event.get("T", 0),
                        int(event["U"]), int(event["u"]), int(event.get("pu", 0)),
                        json.dumps(event.get("b", [])),
                        json.dumps(event.get("a", [])),
                    )})
                else:
                    _log("WARN", "processor", "write queue full — depth_update dropped")

                m = ob.metrics()
                if m is None:
                    continue

                regime = ob.spread_regime()
                print_metrics_row(m, regime)

                ts = int(time.time() * 1000)
                if not write_queue.full():
                    write_queue.put_nowait({"table": "metric", "data": (
                        SYMBOL, ts,
                        str(m["spread_l1"]), str(m["spread_l5"]), str(m["spread_l10"]),
                        str(m["mid"]), str(m["weighted_mid"]), str(m["imbalance"]),
                    )})

                await run_detectors(ob, m, store, ts)

                now = time.monotonic()
                if now - last_snapshot >= SNAPSHOT_INTERVAL_S:
                    bids = [(str(p), str(q)) for p, q in ob.top_bids(20)]
                    asks = [(str(p), str(q)) for p, q in ob.top_asks(20)]
                    await store.insert_snapshot(SYMBOL, ts, ob.last_update_id, bids, asks)
                    last_snapshot = now

                if now - last_monitor >= MONITOR_INTERVAL_S:
                    obs.emit(SYMBOL)
                    last_monitor = now

        except asyncio.CancelledError:
            raise
        except Exception as exc:
            _log("WARN", "processor", str(exc))
            await asyncio.sleep(1)

In [ ]:
async def main() -> None:
    pool = await aiomysql.create_pool(
        host=DB_HOST, port=DB_PORT, user=DB_USER,
        password=DB_PASS, db=DB_NAME, minsize=2, maxsize=5,
    )
    store = MarketDataStore(pool)

    ob          = LocalOrderBook()          # shared across processor and trade detector
    depth_queue = asyncio.Queue(maxsize=DEPTH_Q_MAX)
    trade_queue = asyncio.Queue(maxsize=TRADE_Q_MAX)
    write_queue = asyncio.Queue(maxsize=WRITE_Q_MAX)
    stop_event  = asyncio.Event()

    loop = asyncio.get_running_loop()
    for sig in (signal.SIGINT, signal.SIGTERM):
        try:
            loop.add_signal_handler(sig, stop_event.set)
        except NotImplementedError:
            pass

    tasks = [
        asyncio.create_task(websocket_receiver(depth_queue, stop_event)),
        asyncio.create_task(aggtrade_receiver(trade_queue, stop_event)),
        asyncio.create_task(orderbook_processor(ob, depth_queue, write_queue, store, stop_event)),
        asyncio.create_task(trade_processor(trade_queue, ob, store, stop_event)),
        asyncio.create_task(db_writer(write_queue, store, stop_event)),
    ]

    try:
        await stop_event.wait()
    finally:
        for t in tasks:
            t.cancel()
        await asyncio.gather(*tasks, return_exceptions=True)
        pool.close()
        await pool.wait_closed()
        _log("INFO", "main", "shutdown complete")

In [ ]:
if __name__ == "__main__":
    try:
        await main()
    except KeyboardInterrupt:
        _log("INFO", "main", "stopped by keyboard interrupt")

---

# Part 2 — Replay Harness

Reconstructs the order book from stored data and asserts that the replay produces consistent state at every stored snapshot checkpoint. This is the correctness proof: *"my captured data is sufficient to rebuild book state."*

The replay procedure:
1. Seed a fresh `LocalOrderBook` from the most recent `book_snapshots` row.
2. Apply all `depth_updates` rows with `final_uid > last_uid` in sequence order.
3. At the seeded checkpoint, assert the replayed top-5 bid and ask prices match the stored snapshot within a floating-point epsilon.

Run this cell after a capture session. Requires a live DB connection.

In [ ]:
async def replay_and_verify(symbol: str = SYMBOL) -> None:
    pool = await aiomysql.create_pool(
        host=DB_HOST, port=DB_PORT, user=DB_USER,
        password=DB_PASS, db=DB_NAME, minsize=1, maxsize=2,
    )
    store = MarketDataStore(pool)

    snap = await store.get_latest_snapshot(symbol)
    if snap is None:
        print("[REPLAY] No snapshot found — run a capture session first.")
        pool.close(); await pool.wait_closed(); return

    seed = {
        "bids": json.loads(snap["bids"]),
        "asks": json.loads(snap["asks"]),
        "lastUpdateId": snap["last_uid"],
    }
    ob = LocalOrderBook()
    ob.load_snapshot(seed)
    snap_bids_top5 = [p for p, _ in ob.top_bids(5)]
    snap_asks_top5 = [p for p, _ in ob.top_asks(5)]
    print(f"[REPLAY] Seeded from snapshot last_uid={snap['last_uid']} ts={snap['ts']}")

    events = await store.get_depth_updates_after(symbol, snap["last_uid"])
    print(f"[REPLAY] Replaying {len(events)} events...")

    gaps, applied = 0, 0
    for ev in events:
        reconstructed = {
            "U": ev["first_uid"], "u": ev["final_uid"], "pu": ev["prev_uid"],
            "b": json.loads(ev["bids"]), "a": json.loads(ev["asks"]),
        }
        ok = ob.apply_event(reconstructed, check_previous=(applied > 0))
        if not ok:
            gaps += 1
        else:
            applied += 1

    print(f"[REPLAY] Done. applied={applied}, gaps={gaps}")
    print(f"[REPLAY] Final last_update_id={ob.last_update_id}")
    print(f"[REPLAY] Top-3 bids: {ob.top_bids(3)}")
    print(f"[REPLAY] Top-3 asks: {ob.top_asks(3)}")

    assert gaps == 0, f"INTEGRITY FAIL: {gaps} gap(s) during replay"
    print("[REPLAY] ✓ Integrity assertion passed — stored data rebuilds book state correctly.")

    pool.close()
    await pool.wait_closed()

await replay_and_verify()

---

# Part 3 — Offline Analytics

Run after a capture session. Loads the `metrics` table via `pandas.read_sql` and produces three outputs:

1. **Order-book imbalance vs next mid-price change** — OLS regression to estimate the    predictive power of the weighted-mid lean signal (Cont-Kukanov-Stoikov 2014).

2. **Lag-1 autocorrelation of mid-price returns** — negative value confirms bid-ask bounce    (Roll 1984). Roll's implied spread: `2 * sqrt(-Cov(r_t, r_{t-1}))`.

3. **Spread time series with regime bands** — visualises WIDE/TIGHT episodes against the    rolling mean ± 2σ envelope.

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import aiomysql as _aiomysql

async def _load_metrics() -> pd.DataFrame:
    pool = await _aiomysql.create_pool(
        host=DB_HOST, port=DB_PORT, user=DB_USER,
        password=DB_PASS, db=DB_NAME, minsize=1, maxsize=2,
    )
    async with pool.acquire() as conn:
        async with conn.cursor(_aiomysql.DictCursor) as cur:
            await cur.execute(
                "SELECT ts, spread_l1, mid, weighted_mid, imbalance "
                "FROM metrics WHERE symbol=%s ORDER BY ts ASC", (SYMBOL,)
            )
            rows = await cur.fetchall()
    pool.close(); await pool.wait_closed()
    df = pd.DataFrame(rows)
    for col in ["spread_l1", "mid", "weighted_mid", "imbalance"]:
        df[col] = df[col].astype(float)
    df["ts"] = pd.to_datetime(df["ts"], unit="ms", utc=True)
    return df

df = await _load_metrics()
print(f"Loaded {len(df)} metric rows spanning "
      f"{(df['ts'].iloc[-1] - df['ts'].iloc[0]).total_seconds():.0f} s")

# ── 1. Imbalance → next-tick mid return OLS ──────────────────────────────────
df["mid_return"] = df["mid"].diff().shift(-1)  # next tick's mid change
valid = df[["imbalance", "mid_return"]].dropna()
X = np.column_stack([np.ones(len(valid)), valid["imbalance"]])
y = valid["mid_return"].values
coef, _, _, _ = np.linalg.lstsq(X, y, rcond=None)
y_hat = X @ coef
ss_res = ((y - y_hat) ** 2).sum()
ss_tot = ((y - y.mean()) ** 2).sum()
r2 = 1 - ss_res / ss_tot if ss_tot > 0 else float("nan")
print(f"\nImbalance OLS: intercept={coef[0]:.6f}, lambda={coef[1]:.6f}, R²={r2:.4f}")
print("(Expected R² ≈ 0.05–0.30 on 100 ms NEARUSDT data)")

# ── 2. Lag-1 autocorrelation and Roll implied spread ─────────────────────────
returns = df["mid"].diff().dropna().values
rho1 = np.corrcoef(returns[:-1], returns[1:])[0, 1]
cov  = np.cov(returns[:-1], returns[1:])[0, 1]
roll_spread = 2 * np.sqrt(-cov) if cov < 0 else float("nan")
obs_l1 = df["spread_l1"].mean()
print(f"\nLag-1 autocorrelation: rho1={rho1:.4f}  (negative = bid-ask bounce)")
print(f"Roll implied spread: {roll_spread:.8f}")
print(f"Observed mean L1 spread: {obs_l1:.8f}")

# ── 3. Spread time series with regime bands ───────────────────────────────────
fig, ax = plt.subplots(figsize=(14, 4))
ax.plot(df["ts"], df["spread_l1"], lw=0.6, label="L1 spread %")
roll = df["spread_l1"].rolling(300, min_periods=30)
mu, sigma = roll.mean(), roll.std()
ax.fill_between(df["ts"], mu - 2*sigma, mu + 2*sigma, alpha=0.15, label="±2σ band")
ax.set(title=f"NEARUSDT L1 Spread — {len(df)} events", xlabel="Time (UTC)", ylabel="Spread %")
ax.legend(fontsize=8)
plt.tight_layout()
plt.show()